# 01 — Event register: load, inspect, verify

Goal of this notebook:

1. Load the seeded policy event register
2. Inspect what's in it (counts, date coverage, confidence)
3. Identify events that need verification before the rest of the framework can rely on them
4. Provide a structured workflow for verifying each one

This notebook should be re-run periodically as the register grows.

In [ ]:
import sys
from pathlib import Path

# Make the src package importable
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import pandas as pd
from nem_herding import (
    load_events,
    events_in_window,
    events_for_rez,
    events_of_category,
    events_of_coupling_layer,
    register_summary,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 60)

## Load and summarise

In [ ]:
events = load_events('../data/events/policy_events.csv')
print(f'Loaded {len(events)} events')
print(f'Date range: {events["date"].min()} to {events["date"].max()}')
print(f'Partial-date events: {events["date_is_partial"].sum()}')
print(f'Verified events: {(events["confidence"] == "verified").sum()}')
register_summary(events)

## What needs verifying

Events with `confidence == 'needs_verify'` or partial dates (`XX` in the source CSV).
Work through these systematically — fix the date, replace the source URL with a primary source,
set confidence to `verified`.

In [ ]:
needs_work = events[
    (events['confidence'] == 'needs_verify') | events['date_is_partial']
][['event_id', 'date', 'date_is_partial', 'jurisdiction', 'event_type', 'description', 'source_url']]
needs_work

## Events by REZ

In [ ]:
# How many events affect each REZ (including state-wide and NEM-wide events that affect all)?
rezs = ['CWO', 'NER', 'SWN', 'HCC', 'ILW', 'QLD_NORTH', 'QLD_CENTRAL', 'QLD_SOUTH']
for rez in rezs:
    rez_events = events_for_rez(events, rez)
    rez_specific = rez_events[rez_events['rez'] == rez]
    print(f'{rez:15s}  total relevant: {len(rez_events):3d}  '
          f'rez-specific: {len(rez_specific):3d}')

## Events by coupling layer

The pre-registered prediction is that Layer 2 (informational) events should produce larger
synchronisation responses in mature REZs, while Layer 1 (direct) events dominate in early-stage
REZs. Sanity check: do we have a reasonable mix of both layers in the register?

In [ ]:
events.groupby(['coupling_layer', 'category']).size().unstack(fill_value=0)

## Verification workflow per event

Pick an event_id from the `needs_work` table above. For each:

1. Open the source_url (or search if it's blank)
2. Confirm exact date
3. Edit `data/events/policy_events.csv` directly — set date, source_url, confidence='verified'
4. Reload here and confirm it's no longer in `needs_work`

Suggested batching: do *all NSW REZ designations* in one session, then *all federal CIS events*
in another, then *all AEMO publications*, etc. Within-type batching reuses your context and
is much faster than jumping around.

In [ ]:
# Example: show one event in detail
event_id = 'e004'  # New England REZ declaration - needs date verification
events[events['event_id'] == event_id].T